In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("indian_movies.csv")

In [ ]:
df.shape

(22881, 9)

In [ ]:
df.columns

Index(['id', 'title', 'industry', 'genre', 'overview', 'release_date',
       'rating', 'popularity', 'poster_path'],
      dtype='object')

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df['id'].duplicated().sum()

np.int64(0)

In [ ]:
df.isna().sum()

,0
id,0
title,0
industry,0
genre,3578
overview,1285
release_date,0
rating,0
popularity,0
poster_path,2699


In [ ]:
columns = [
    "genre",
    "overview",
    "poster_path"
]

for col in columns:
    df[col] = df[col].fillna("")

In [ ]:
df.isna().sum()

,0
id,0
title,0
industry,0
genre,0
overview,0
release_date,0
rating,0
popularity,0
poster_path,0


In [ ]:
df["features"] =(
     df["genre"] + " " +
     df["overview"] + " " +
     df["industry"] + " " +
     df["title"] + " "  +
     df["popularity"].astype(str)+ " "+
     df["rating"].astype(str)
     )

In [ ]:
df["features"]

,features
0,"Drama, History 7th century AD. Parthiban, the ..."
1,Drama Kollywood Kuravanji 0.2792 7.0
2,Drama Paadhai Theriyudhu Paar is a 1960 Indian...
3,"Romance, Drama A landlord's son and a farmer's..."
4,"Drama Chandrasekaran, an industrialist, is ver..."
...,...
22876,"Drama, Thriller Daksh deals with his mental ba..."
22877,"Drama At a city fun fair, three strangers an a..."
22878,Action Ten Friends Navigate Their Complex Live...
22879,"Drama, Family Kala kagaz is a short film that ..."


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df["features"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/716 [00:00<?, ?it/s]

In [ ]:
embeddings

array([[-0.01924507, -0.00764642, -0.05994853, ...,  0.05040755,
        -0.03842559, -0.04276317],
       [-0.01204734, -0.03922189, -0.03060419, ...,  0.02910927,
        -0.03899678, -0.00445966],
       [-0.08241191,  0.01088133, -0.05610657, ...,  0.05044236,
         0.00809225,  0.00038371],
       ...,
       [-0.03080192,  0.01637074,  0.0196087 , ...,  0.06228131,
        -0.0722805 , -0.01418586],
       [-0.08487599,  0.02234096, -0.07761329, ..., -0.00218967,
        -0.07645778,  0.00170557],
       [-0.03033092,  0.07418958, -0.06493036, ...,  0.00192617,
        -0.02904047,  0.01066238]], dtype=float32)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(embeddings)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

In [ ]:
def recommend_movies(movie_name, top_n=10):

    movie_name = movie_name.lower()

    matches = df[df["title"].str.lower() == movie_name]

    if matches.empty:
        return "Movie not found."

    idx = matches.index[0]

    similarity_scores = list(enumerate(similarity_matrix[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    for i, score in similarity_scores[1:top_n+1]:

        recommendations.append({
            "Title": df.iloc[i]["title"],
            "Genre": df.iloc[i]["genre"],
            "Rating": df.iloc[i]["rating"],
            "Similarity": round(score,3)
        })

    return pd.DataFrame(recommendations)

In [ ]:
recommend_movies("Dude")

,Title,Genre,Rating,Similarity
0,Kuthu,"Action, Drama, Family",5.000,0.747
1,Nesam,"Drama, Romance",0.000,0.680
2,Kadhal Desam,"Action, Comedy, Drama, Romance",7.313,0.673
3,Kadukka,"Romance, Comedy, Drama",0.000,0.672
4,Kumbaari,"Romance, Comedy",0.000,0.667
5,Hello,"Romance, Drama",6.200,0.666
6,Kallukul Eeram,"Romance, Drama",9.000,0.665
7,Annamalai,"Drama, Romance",7.367,0.665
8,Ullasam,"Action, Romance",5.500,0.664
9,Onbathu Kuzhi Sampath,"Drama, Romance",6.000,0.663


In [ ]:
df['release_date'] = pd.to_datetime(df['release_date'])

In [ ]:
df['release_date']

,release_date
0,1960-06-03
1,1960-03-04
2,1960-11-18
3,1960-03-15
4,1960-06-25
...,...
22876,2026-02-27
22877,2026-01-20
22878,2026-01-04
22879,2026-03-06


In [ ]:
df['year'] = df['release_date'].dt.year

In [ ]:
df['year']

,year
0,1960
1,1960
2,1960
3,1960
4,1960
...,...
22876,2026
22877,2026
22878,2026
22879,2026


In [ ]:
df["genre_list"] = df["genre"].apply(
    lambda x: [g.strip().lower() for g in x.split(",")]
)

In [ ]:
df['genre_list']

,genre_list
0,"[drama, history]"
1,[drama]
2,[drama]
3,"[romance, drama]"
4,[drama]
...,...
22876,"[drama, thriller]"
22877,[drama]
22878,[action]
22879,"[drama, family]"


In [ ]:
def recommend_by_genre_year(
    genre,
    year,
    top_n=10
):

    genre = genre.lower()

    movies = df[
        (df["year"] == year) &
        (df["genre_list"].apply(lambda x: genre in x))
    ]

    if movies.empty:
        return "No movies found."

    movies = movies.sort_values(
        by=["rating","popularity"],
        ascending=False
    )

    return movies[
        [
            "title",
            "genre",
            "rating",
            "popularity",
            "poster_path"
        ]
    ].head(top_n)

In [ ]:
recommend_by_genre_year(
    "Action",
    2022
)

,title,genre,rating,popularity,poster_path
18881,Pyar Mein Thoda Twist,"Romance, Drama, Comedy, Action",10.000,0.1886,https://image.tmdb.org/t/p/w500/ajstjdHlvg4Zme...
18204,HEROIC,"Action, Science Fiction",10.000,0.1271,https://image.tmdb.org/t/p/w500/h40gAj2rtnSahW...
18910,Triple Cross,"Thriller, Action",10.000,0.0935,https://image.tmdb.org/t/p/w500/d9J9Fi9r3LrIeX...
18122,Selfie,"Action, Thriller",8.000,0.3517,https://image.tmdb.org/t/p/w500/7Su5HR6P73lKUZ...
18691,Dear Vikram,"Action, Drama",8.000,0.0922,https://image.tmdb.org/t/p/w500/2AL01qmhWd47YZ...
18241,RRR,"Action, History, Drama",7.715,13.0885,https://image.tmdb.org/t/p/w500/u0XUBNQWlOvrh0...
18053,Vikram,"Action, Crime, Thriller",7.567,3.1781,https://image.tmdb.org/t/p/w500/774UV1aCURb4s4...
18637,K.G.F: Chapter 2,"Action, Thriller, Adventure, Crime",7.376,13.6946,https://image.tmdb.org/t/p/w500/khNVygolU0TxLI...
18432,Salute,"Action, Crime, Thriller",7.167,0.6497,https://image.tmdb.org/t/p/w500/1rTESlmFExGj4Y...
18638,Kantara,"Action, Drama, Thriller",7.142,3.1692,https://image.tmdb.org/t/p/w500/jIsKmkxMzdCZ0U...


In [ ]:
import joblib

model_file_path = '/content/movie_recommendation_model.pkl'

model_components = {
    'dataframe': df,
    'embeddings': embeddings,
}

joblib.dump(model_components, model_file_path)

print(f"All model components and data saved to {model_file_path}")

All model components and data saved to /content/movie_recommendation_model.pkl
